# Lesson 11: k-Nearest Neighbours

*ML Academy — companion notebook*

Classify by asking the neighbours: distances, choosing k, scaling and the curse of dimensionality.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-11.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 10. In Python

In [1]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
print("without scaling:", round(raw.score(X_test, y_test), 3))

scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)
print("with scaling:   ", round(scaled.score(X_test, y_test), 3))

for k in [1, 5, 15, 51, 81, 101]:
    s = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)),
                        X_train, y_train, cv=5)
    print(f"k={k:3d}  CV accuracy {s.mean():.3f} ± {s.std():.3f}")

without scaling: 0.667
with scaling:    0.933
k=  1  CV accuracy 0.918 ± 0.054
k=  5  CV accuracy 0.940 ± 0.038
k= 15  CV accuracy 0.932 ± 0.037
k= 51  CV accuracy 0.962 ± 0.023
k= 81  CV accuracy 0.593 ± 0.065


k=101  CV accuracy 0.399 ± 0.019


## 13. Going deeper: distances, dimensions and fast neighbour search

In [2]:
import numpy as np

# Distance concentration: in high dimensions the nearest and farthest points are almost equally far
rng = np.random.default_rng(0)
for d in [2, 10, 100, 1000]:
    pts = rng.uniform(size=(1000, d)); q = rng.uniform(size=d)
    dist = np.linalg.norm(pts - q, axis=1)
    print(f"{d:4d} dimensions: nearest {dist.min():6.2f}   farthest {dist.max():6.2f}   ratio {dist.max() / dist.min():5.2f}")

   2 dimensions: nearest   0.01   farthest   1.33   ratio 135.62
  10 dimensions: nearest   0.47   farthest   2.11   ratio  4.48
 100 dimensions: nearest   3.35   farthest   4.65   ratio  1.39
1000 dimensions: nearest  12.17   farthest  13.55   ratio  1.11


In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
for metric in ["euclidean", "manhattan", "cosine"]:
    for weights in ["uniform", "distance"]:
        m = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15, metric=metric, weights=weights))
        print(f"{metric:9} {weights:8}: accuracy {cross_val_score(m, X, y, cv=10).mean():.3f}")

euclidean uniform : accuracy 0.961
euclidean distance: accuracy 0.963


manhattan uniform : accuracy 0.954
manhattan distance: accuracy 0.956


cosine    uniform : accuracy 0.963
cosine    distance: accuracy 0.963


In [4]:
import time
from sklearn.neighbors import NearestNeighbors

data = rng.normal(size=(200000, 3)); queries = rng.normal(size=(2000, 3))
found, times = {}, {}
for algo in ["brute", "kd_tree"]:
    nn = NearestNeighbors(n_neighbors=5, algorithm=algo).fit(data)
    t = time.perf_counter(); found[algo] = nn.kneighbors(queries)[1]; times[algo] = time.perf_counter() - t
print("same neighbours found by both:", np.array_equal(np.sort(found["brute"]), np.sort(found["kd_tree"])))
print("k-d tree faster:", times["kd_tree"] < times["brute"])   # exact timings vary from computer to computer

same neighbours found by both: True
k-d tree faster: True


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [5]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
cols = ["size_m2", "dist_km", "age_yrs"]
knn = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=3)).fit(df[cols], df["price_lakh"])
new = pd.DataFrame({"size_m2": [100], "dist_km": [6], "age_yrs": [5]})
dist, idx = knn[-1].kneighbors(knn[0].transform(new))
print("nearest houses:", (df["id"].iloc[idx[0]]).tolist(), "prices:", df["price_lakh"].iloc[idx[0]].tolist())
print("predicted price:", round(knn.predict(new)[0], 1), "lakh")

nearest houses: [5, 4, 2] prices: [88, 55, 82]
predicted price: 75.0 lakh


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [6]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Distances to every point

Compute the Euclidean distance from q to each row of X in one line.

In [7]:
import numpy as np
X = np.array([[0, 0], [1, 1], [4, 4], [5, 4], [1, 4.0]])
q = np.array([2, 1.0])
d = None  # ← your code here
check(d, [2.23606797749979, 1.0, 3.605551275463989, 4.242640687119285, 3.1622776601683795], hint='Subtract q from every row, square, sum along axis=1, square root.')

❌ Not yet. Hint: Subtract q from every row, square, sum along axis=1, square root.


### Challenge 2: Indices of the 3 nearest

Find the indices of the 3 nearest points, nearest first.

In [8]:
import numpy as np
d = np.array([2.24, 1.0, 3.61, 4.24, 3.16])
nearest = None  # ← your code here
check(nearest, [1, 0, 4], hint='np.argsort gives the order from smallest to largest; keep the first 3.')

❌ Not yet. Hint: np.argsort gives the order from smallest to largest; keep the first 3.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Distances to every point** — the missing line:

```python
d = np.sqrt(((X - q) ** 2).sum(axis=1))
```

**2. Indices of the 3 nearest** — the missing line:

```python
nearest = np.argsort(d)[:3]
```